In [2]:
import pandas as pd
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
import pickle
from tqdm import tqdm

c:\Users\Yazan\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# === Load CSV ===
csv_path = "..\datasets\cleaned_movie_plots_v2.csv"
df = pd.read_csv(csv_path)


<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Yazan\AppData\Local\Temp\ipykernel_11848\1647294328.py:2: SyntaxWarning: invalid escape sequence '\d'
  csv_path = "..\datasets\cleaned_movie_plots_v2.csv"


In [4]:
# === Build combined text for embeddings ===
def build_embedding_text(row):
    parts = [
        f"Title: {row['title']}",
        f"Release year: {row['release_year']}",
    ]
    if str(row['genre']).strip().lower() != "unknown":
        parts.append(f"Genre: {row['genre']}")
    parts.append(f"Summary: {row['summary']}")
    parts.append(f"Plot: {row['plot']}")
    return " ".join(parts)

texts = df.apply(build_embedding_text, axis=1).tolist()


In [5]:
# Load model and tokenizer
from sentence_transformers import SentenceTransformer

model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3433.31it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
# === Encoding Function ===
embeddings = model.encode(
    texts,
    batch_size=32,  # you can tweak this depending on your CPU/RAM
    show_progress_bar=True
)

Batches: 100%|██████████| 1012/1012 [1:05:48<00:00,  3.90s/it]


In [7]:
# === Save embeddings ===
np.save("movie_embeddings_2.npy", embeddings)

In [8]:
# === Save metadata ===
metadata = df.drop(columns=["plot"]).to_dict(orient="records")
with open("movie_metadata_2.pkl", "wb") as f:
    pickle.dump(metadata, f)